In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# C19 — Ablation du débruitage (D4)\n",
    "\n",
    "D4 impose de **mesurer** le débruitage avant de l'activer. On compare quatre forces : **sans**,\n",
    "**0,5**, **0,8** et **1,0** (`prop_decrease` du spectral gating, seuil fixé à 1,5), sur la\n",
    "**validation**, dans quatre conditions : extrait propre, puis bruité à **20, 10 et 0 dB** de\n",
    "rapport signal/bruit (bruit mélangé au niveau exact).\n",
    "\n",
    "Deux mesures, avec intervalles à 95 % par bootstrap (C18) :\n",
    "\n",
    "- **WER** de faster-whisper (C17) sur la partie `dev` d'OpenSLR 57 (986 phrases) ;\n",
    "- **F1 macro des émotions** du meilleur modèle (C11 : tête sur Whisper base), sur la partie\n",
    "  « validation » de CREMA-D + RAVDESS (P1).\n",
    "\n",
    "**Règle de D4** : on retient la force au meilleur score moyen ; à égalité, la plus faible.\n",
    "\n",
    "Choix de protocole (à relire par Lynn) :\n",
    "- le bruit vient de classes d'ESC-50 au fond stable (`configs/denoise_ablation.yaml`) ; un même\n",
    "  extrait de bruit est utilisé pour toutes les forces : les comparaisons sont appariées ;\n",
    "- chaîne : extrait (mono 16 kHz, D2) → mélange au SNR → débruitage → modèle. Pas de\n",
    "  normalisation RMS (D3) ici, comme en C10 et C17 : la condition « sans » + « propre » doit\n",
    "  retrouver l'ordre de grandeur de C11 et de C17 (contrôle de la section g) ;\n",
    "- score moyen d'une force = moyenne des deux tâches, chacune moyennée sur les 4 conditions ;\n",
    "  le score de transcription est 1 − WER (plancher 0), pour que « plus haut = mieux » partout."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## a. Démarrage\n",
    "\n",
    "On clone le dépôt, on l'installe en mode éditable (I9), puis faster-whisper, CTranslate2 et jiwer (versions de C17), et cuBLAS/cuDNN pour CUDA 12. GPU T4, Internet activé."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "import sys\n",
    "\n",
    "BRANCH = \"exp/c19-denoising-ablation\"\n",
    "REPO_URL = \"https://github.com/mentalist011101/TP_SIGNAL_DS.git\"\n",
    "CLONE_DIR = \"/kaggle/working/TP_SIGNAL_DS\"\n",
    "\n",
    "!rm -rf {CLONE_DIR}\n",
    "!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}\n",
    "# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.\n",
    "%pip install -q --ignore-requires-python -e {CLONE_DIR}\n",
    "%pip install -q \"faster-whisper==1.2.1\" \"ctranslate2==4.8.5\" \"jiwer==4.0.0\"\n",
    "%pip install -q \"nvidia-cublas-cu12==12.9.3.13\" \"nvidia-cudnn-cu12==9.28.1.4\"\n",
    "\n",
    "sys.path.insert(0, CLONE_DIR + \"/src\")\n",
    "print(sys.version)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## b. ffmpeg, GPU et bibliothèques CUDA 12\n",
    "\n",
    "Même préparation qu'en C17 (f) : cuBLAS et cuDNN chargés par leur chemin avant le premier calcul de CTranslate2."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "import ctypes\n",
    "import importlib.util\n",
    "import shutil\n",
    "import time\n",
    "from pathlib import Path\n",
    "\n",
    "import ctranslate2\n",
    "import torch\n",
    "\n",
    "if shutil.which(\"ffmpeg\") is None:\n",
    "    !apt-get -qq update && apt-get -qq install -y ffmpeg\n",
    "NOTEBOOK_START = time.time()\n",
    "\n",
    "\n",
    "def nvidia_library_dir(package):\n",
    "    spec = importlib.util.find_spec(package)\n",
    "    return Path(spec.submodule_search_locations[0]) / \"lib\"\n",
    "\n",
    "\n",
    "def preload_libraries(directory):\n",
    "    # Chargement global, en plusieurs passes : une bibliothèque peut dépendre d'une autre.\n",
    "    pending = sorted(directory.glob(\"lib*.so*\"))\n",
    "    for _ in range(len(pending)):\n",
    "        failed = []\n",
    "        for path in pending:\n",
    "            try:\n",
    "                ctypes.CDLL(str(path), mode=ctypes.RTLD_GLOBAL)\n",
    "            except OSError:\n",
    "                failed.append(path)\n",
    "        if len(failed) == len(pending):\n",
    "            break\n",
    "        pending = failed\n",
    "    return [path.name for path in pending]\n",
    "\n",
    "\n",
    "if ctranslate2.get_cuda_device_count() == 0 or not torch.cuda.is_available():\n",
    "    raise RuntimeError(\"Aucun GPU : activer l'accélérateur GPU T4.\")\n",
    "for package in [\"nvidia.cublas\", \"nvidia.cudnn\"]:\n",
    "    print(package, \"non chargées :\", preload_libraries(nvidia_library_dir(package)))\n",
    "DEVICE = torch.device(\"cuda\")\n",
    "print(\"GPU :\", torch.cuda.get_device_name(0))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## c. Jeux de données\n",
    "\n",
    "CREMA-D et RAVDESS (datasets Kaggle attachés), ESC-50 (bruits ; archive officielle) et OpenSLR 57\n",
    "(archive officielle, ~1,8 Go, décompressée dans `/tmp`). Liens dans `/kaggle/working/datasets/<id>`\n",
    "(contrat des manifestes, `docs/datasets.md`)."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "import tarfile\n",
    "import urllib.request\n",
    "import zipfile\n",
    "\n",
    "INPUT_DIR = Path(\"/kaggle/input\")\n",
    "DATASETS_DIR = Path(\"/kaggle/working/datasets\")\n",
    "DOWNLOAD_DIR = Path(\"/kaggle/working/downloads\")\n",
    "OPENSLR_DIR = Path(\"/tmp/openslr57\")\n",
    "ESC50_ARCHIVE_URL = \"https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip\"\n",
    "OPENSLR57_TAR_URL = \"https://openslr.elda.org/resources/57/African_Accented_French.tar.gz\"\n",
    "CREMAD_EXPECTED_WAV = 7442\n",
    "RAVDESS_EXPECTED_WAV = 1440\n",
    "\n",
    "\n",
    "def find_cremad_root():\n",
    "    for audio_folder in sorted(INPUT_DIR.glob(\"**/AudioWAV\")):\n",
    "        if len(list(audio_folder.glob(\"*.wav\"))) == CREMAD_EXPECTED_WAV:\n",
    "            return audio_folder.parent\n",
    "    raise RuntimeError(\"CREMA-D introuvable : attacher le dataset ejlok1/cremad.\")\n",
    "\n",
    "\n",
    "def find_ravdess_root():\n",
    "    for candidate in sorted({actor.parent for actor in INPUT_DIR.glob(\"**/Actor_01\")}):\n",
    "        wav_paths = list(candidate.glob(\"Actor_*/*.wav\"))\n",
    "        if len(wav_paths) == RAVDESS_EXPECTED_WAV:\n",
    "            return candidate\n",
    "    raise RuntimeError(\"RAVDESS introuvable : attacher uwrfkaggler/ravdess-emotional-speech-audio.\")\n",
    "\n",
    "\n",
    "def download_esc50():\n",
    "    found = sorted(INPUT_DIR.glob(\"**/meta/esc50.csv\"))\n",
    "    if found:\n",
    "        return found[0].parent.parent\n",
    "    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)\n",
    "    archive_path = DOWNLOAD_DIR / \"esc50.zip\"\n",
    "    urllib.request.urlretrieve(ESC50_ARCHIVE_URL, archive_path)\n",
    "    with zipfile.ZipFile(archive_path) as archive:\n",
    "        archive.extractall(DOWNLOAD_DIR)\n",
    "    archive_path.unlink()\n",
    "    return DOWNLOAD_DIR / \"ESC-50-master\"\n",
    "\n",
    "\n",
    "def download_openslr57():\n",
    "    OPENSLR_DIR.mkdir(parents=True, exist_ok=True)\n",
    "    archive_path = OPENSLR_DIR / \"openslr57.tar.gz\"\n",
    "    urllib.request.urlretrieve(OPENSLR57_TAR_URL, archive_path)\n",
    "    with tarfile.open(archive_path) as archive:\n",
    "        archive.extractall(OPENSLR_DIR, filter=\"data\")\n",
    "    archive_path.unlink()\n",
    "    return sorted(OPENSLR_DIR.glob(\"**/African_Accented_French\"))[0]\n",
    "\n",
    "\n",
    "roots = {\n",
    "    \"esc50\": download_esc50(),\n",
    "    \"cremad\": find_cremad_root(),\n",
    "    \"ravdess\": find_ravdess_root(),\n",
    "    \"openslr57\": download_openslr57(),\n",
    "}\n",
    "DATASETS_DIR.mkdir(parents=True, exist_ok=True)\n",
    "for dataset_id, root in roots.items():\n",
    "    link = DATASETS_DIR / dataset_id\n",
    "    if link.is_symlink():\n",
    "        link.unlink()\n",
    "    link.symlink_to(root)\n",
    "    print(dataset_id, \"→\", root)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## d. Configuration et jeux de validation\n",
    "\n",
    "Les paramètres sont dans `configs/denoise_ablation.yaml` (aucun nombre dans le code). On lit :\n",
    "la validation des émotions (découpage P1 versionné, classes communes P2) et la partie `dev`\n",
    "d'OpenSLR 57. On vérifie que chaque audio existe."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "\n",
    "from echolens.config import CONFIG_DIR, REPO_ROOT\n",
    "from echolens.core.audio import load_audio\n",
    "from echolens.core.audio.preprocess_config import load_preprocess_config\n",
    "from echolens.core.data.emotions import load_emotion_labels\n",
    "from echolens.core.data.manifest import audio_path, load_manifest\n",
    "from echolens.core.data.splits import check_no_speaker_leak, load_speaker_splits, load_split_config\n",
    "from echolens.training.denoise_ablation import load_ablation_config\n",
    "from echolens.training.evaluation import EvaluationConfig\n",
    "from echolens.training.heads import emotion_examples, load_heads_config\n",
    "from echolens.training.transcription_eval import (\n",
    "    CorpusSelection,\n",
    "    corpus_rows,\n",
    "    load_transcription_eval_config,\n",
    "    sample_rows,\n",
    ")\n",
    "\n",
    "ablation = load_ablation_config()\n",
    "evaluation_config = EvaluationConfig.from_yaml(CONFIG_DIR / \"evaluation.yaml\")\n",
    "transcription_config = load_transcription_eval_config()\n",
    "base_denoise_config = load_preprocess_config().denoise\n",
    "labels = load_emotion_labels()\n",
    "split_config = load_split_config()\n",
    "speaker_splits = load_speaker_splits(REPO_ROOT / \"data\" / \"splits\" / \"emotion_split.csv\")\n",
    "check_no_speaker_leak(speaker_splits)\n",
    "manifests_dir = REPO_ROOT / \"data\" / \"manifests\"\n",
    "\n",
    "# Étape 1 : extraits d'émotions de la validation, dans l'ordre des manifestes.\n",
    "emotion_clips = []\n",
    "for dataset_id in load_heads_config().emotion_datasets:\n",
    "    rows = load_manifest(manifests_dir / f\"{dataset_id}.csv\")\n",
    "    for example in emotion_examples(rows, labels, speaker_splits, split_config):\n",
    "        if example.split == \"validation\":\n",
    "            emotion_clips.append((rows[example.position], example.target))\n",
    "print(\"Émotions, validation :\", len(emotion_clips), \"extraits\")\n",
    "\n",
    "# Étape 2 : phrases de la partie dev d'OpenSLR 57.\n",
    "openslr_rows = load_manifest(manifests_dir / \"openslr57.csv\")\n",
    "dev_rows = corpus_rows(openslr_rows, CorpusSelection(id=\"dev\", official_split=\"dev\"))\n",
    "if ablation.wer_sentences is not None:\n",
    "    dev_rows = sample_rows(dev_rows, ablation.wer_sentences, ablation.seed)\n",
    "print(\"Transcription, dev :\", len(dev_rows), \"phrases,\", len({row.speaker for row in dev_rows}), \"locuteurs\")\n",
    "\n",
    "# Étape 3 : tous les audios sont là.\n",
    "all_rows = [row for row, _ in emotion_clips] + dev_rows\n",
    "missing = [row.path for row in all_rows if not audio_path(row, DATASETS_DIR).exists()]\n",
    "if missing:\n",
    "    raise RuntimeError(f\"{len(missing)} audios absents, par exemple {missing[:3]}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## e. Bruits, SNR exact et chaîne de traitement\n",
    "\n",
    "Les extraits de bruit sont les classes `noise_classes` d'ESC-50 (rain, wind, vacuum_cleaner, …),\n",
    "lus avec `load_audio`. Pour l'extrait numéro *i*, le générateur de graine `seed + i` tire un bruit\n",
    "et un point de départ : **le bruit est le même pour toutes les forces**. Le gain du bruit est\n",
    "calculé pour que le SNR soit exact (`mix_at_snr`, testé à la main dans\n",
    "`tests/unit/test_denoise_ablation.py`). Vérification : le SNR mesuré doit valoir 20, 10 et 0 dB."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "from echolens.core.audio.denoise import denoise\n",
    "from echolens.training.denoise_ablation import (\n",
    "    fit_noise_length,\n",
    "    measured_snr_db,\n",
    "    mix_at_snr,\n",
    "    pick_noise_clip,\n",
    ")\n",
    "\n",
    "# Étape 1 : le lot de bruits d'ESC-50.\n",
    "esc50_rows = load_manifest(manifests_dir / \"esc50.csv\")\n",
    "noise_pool = []\n",
    "for row in esc50_rows:\n",
    "    if row.label in ablation.noise_classes:\n",
    "        noise_pool.append(load_audio(audio_path(row, DATASETS_DIR)))\n",
    "print(\"Bruits :\", len(noise_pool), \"extraits de\", ablation.noise_classes)\n",
    "\n",
    "\n",
    "def noise_for_clip(clip_index, length):\n",
    "    # Étape 2 : bruit de la bonne longueur, toujours le même pour un extrait donné.\n",
    "    rng = np.random.default_rng(ablation.seed + clip_index)\n",
    "    noise = pick_noise_clip(noise_pool, ablation.noise_min_dbfs, ablation.noise_max_tries, rng)\n",
    "    start = int(rng.integers(0, len(noise)))\n",
    "    return fit_noise_length(noise, length, start)\n",
    "\n",
    "\n",
    "def denoise_config_for(strength):\n",
    "    # Force None = « sans » : enabled False, le signal est renvoyé inchangé (denoise.py).\n",
    "    if strength is None:\n",
    "        return base_denoise_config.model_copy(update={\"enabled\": False})\n",
    "    return base_denoise_config.model_copy(update={\"enabled\": True, \"prop_decrease\": strength})\n",
    "\n",
    "\n",
    "def prepare(signal, clip_index, snr_db, strength):\n",
    "    # Étape 3 : mélange au SNR (sauf extrait propre), puis débruitage.\n",
    "    if snr_db is not None:\n",
    "        noise = noise_for_clip(clip_index, len(signal))\n",
    "        signal = mix_at_snr(signal, noise, snr_db)\n",
    "    return denoise(signal, denoise_config_for(strength))\n",
    "\n",
    "\n",
    "# Contrôle du SNR sur un extrait de parole.\n",
    "probe_signal = load_audio(audio_path(dev_rows[0], DATASETS_DIR))\n",
    "for snr in [20, 10, 0]:\n",
    "    mixed = prepare(probe_signal, 0, snr, None)\n",
    "    print(f\"SNR demandé {snr:2d} dB → mesuré {measured_snr_db(probe_signal, mixed):.3f} dB\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## f. Mesures\n",
    "\n",
    "Une **cellule** = une force × une condition (16 au total). Les résultats de chaque cellule sont\n",
    "écrits dans `c19_cells.json` dès qu'elle est finie : si Kaggle coupe la session, une relance (avec\n",
    "cette sortie ajoutée en entrée du notebook) reprend où le calcul s'était arrêté.\n",
    "\n",
    "- Émotions : encodeur Whisper base gelé (sur GPU) → moyenne des couches sur les trames utiles →\n",
    "  standardisation de l'entraînement (R4) → tête du registre (poids à la révision figée) → classe.\n",
    "  Même code que `ModelPredictor`, mais l'encodeur tourne sur GPU.\n",
    "- Transcription : faster-whisper float16, réglages de `configs/transcription.yaml`, normalisation\n",
    "  P12, WER avec intervalle par bootstrap **par locuteur** (C17)."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "import json\n",
    "\n",
    "from huggingface_hub import hf_hub_download\n",
    "\n",
    "from echolens.core.models.heads import LayerWeightedHead\n",
    "from echolens.core.models.production_config import CONFIG_FILE, WEIGHTS_FILE, load_production_config\n",
    "from echolens.core.models.registry import ModelRegistry\n",
    "from echolens.core.models.storage import load_weights\n",
    "from echolens.core.models.whisper_encoder import load_whisper_encoder, whisper_useful_layer_means\n",
    "\n",
    "# Étape 1 : tête des émotions, à la révision figée du registre (configs/registry.yaml).\n",
    "registry = ModelRegistry.from_yaml(CONFIG_DIR / \"registry.yaml\")\n",
    "entry = [model for model in registry.models if model.id == ablation.emotion_model_id][0]\n",
    "repo_id = entry.weights.removeprefix(\"hf://\")\n",
    "model_config = load_production_config(Path(hf_hub_download(repo_id, CONFIG_FILE, revision=entry.revision)))\n",
    "weights_path = Path(hf_hub_download(repo_id, WEIGHTS_FILE, revision=entry.revision))\n",
    "if model_config.labels != [label.id for label in labels.emotions]:\n",
    "    raise RuntimeError(\"Classes du modèle différentes de configs/labels.yaml\")\n",
    "shape = model_config.head\n",
    "head = LayerWeightedHead(\n",
    "    shape.layer_count, shape.dimension, shape.hidden_size, len(model_config.labels), shape.dropout\n",
    ")\n",
    "head = load_weights(head, weights_path)\n",
    "head_mean = np.asarray(model_config.standardization_mean, dtype=np.float32)\n",
    "head_std = np.asarray(model_config.standardization_std, dtype=np.float32)\n",
    "\n",
    "# Étape 2 : encodeur Whisper gelé, sur GPU.\n",
    "reference = model_config.encoder\n",
    "encoder = load_whisper_encoder(\n",
    "    reference.checkpoint,\n",
    "    reference.revision,\n",
    "    reference.sample_rate,\n",
    "    reference.samples_per_output_frame,\n",
    "    reference.max_output_frames,\n",
    "    DEVICE,\n",
    ")\n",
    "print(entry.id, \"· révision\", entry.revision[:8], \"· F1 test du registre\", entry.metrics)\n",
    "\n",
    "\n",
    "def predict_emotion(signal):\n",
    "    means = whisper_useful_layer_means(signal, encoder)\n",
    "    standardized = ((means - head_mean) / head_std).astype(np.float32)\n",
    "    inputs = torch.from_numpy(standardized).unsqueeze(0)\n",
    "    with torch.no_grad():\n",
    "        scores = head(inputs)\n",
    "    return int(scores[0].argmax())"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "from echolens.training.transcription_eval import score_sentence\n",
    "from echolens.training.transcription_models import load_whisper, transcribe\n",
    "\n",
    "whisper_model_config = [m for m in transcription_config.models if m.id == ablation.whisper_model_id][0]\n",
    "whisper = load_whisper(whisper_model_config, \"cuda\", \"float16\", transcription_config)\n",
    "print(\"Whisper\", whisper_model_config.id, \"chargé sur GPU, révision\", whisper_model_config.revision[:8])"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "OUTPUT_DIR = Path(\"/kaggle/working\")\n",
    "CELLS_PATH = OUTPUT_DIR / \"c19_cells.json\"\n",
    "\n",
    "\n",
    "def cell_key(strength, snr_db):\n",
    "    strength_name = \"none\" if strength is None else str(strength)\n",
    "    snr_name = \"clean\" if snr_db is None else str(snr_db)\n",
    "    return f\"{strength_name}|{snr_name}\"\n",
    "\n",
    "\n",
    "# Reprise : cellules déjà calculées (sortie d'un passage précédent, ajoutée en entrée).\n",
    "cells = {}\n",
    "for previous in sorted(INPUT_DIR.glob(\"**/c19_cells.json\")):\n",
    "    if not CELLS_PATH.exists():\n",
    "        shutil.copy(previous, CELLS_PATH)\n",
    "if CELLS_PATH.exists():\n",
    "    cells = json.loads(CELLS_PATH.read_text(encoding=\"utf-8\"))\n",
    "print(len(cells), \"cellules déjà calculées sur\", len(ablation.strengths) * len(ablation.snr_db))\n",
    "\n",
    "\n",
    "def run_emotion_cell(strength, snr_db):\n",
    "    predictions = []\n",
    "    for clip_index, (row, _) in enumerate(emotion_clips):\n",
    "        signal = load_audio(audio_path(row, DATASETS_DIR))\n",
    "        prepared = prepare(signal, clip_index, snr_db, strength)\n",
    "        predictions.append(predict_emotion(prepared))\n",
    "    return predictions\n",
    "\n",
    "\n",
    "def run_wer_cell(strength, snr_db):\n",
    "    hypotheses = []\n",
    "    for clip_index, row in enumerate(dev_rows):\n",
    "        signal = load_audio(audio_path(row, DATASETS_DIR))\n",
    "        # Indices décalés : le bruit d'une phrase ne reprend pas celui d'un extrait d'émotion.\n",
    "        prepared = prepare(signal, 100_000 + clip_index, snr_db, strength)\n",
    "        hypotheses.append(transcribe(whisper, prepared, transcription_config))\n",
    "    return hypotheses\n",
    "\n",
    "\n",
    "for strength in ablation.strengths:\n",
    "    for snr_db in ablation.snr_db:\n",
    "        key = cell_key(strength, snr_db)\n",
    "        if key in cells:\n",
    "            continue\n",
    "        start = time.perf_counter()\n",
    "        emotion_predictions = run_emotion_cell(strength, snr_db)\n",
    "        hypotheses = run_wer_cell(strength, snr_db)\n",
    "        cells[key] = {\"emotion_predictions\": emotion_predictions, \"hypotheses\": hypotheses}\n",
    "        CELLS_PATH.write_text(json.dumps(cells, ensure_ascii=False), encoding=\"utf-8\")\n",
    "        print(f\"{key:12s} fait en {time.perf_counter() - start:.0f} s\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## g. Scores, intervalles et contrôle\n",
    "\n",
    "Pour chaque cellule : F1 macro des émotions avec intervalle (`bootstrap_ci`, C18), WER avec\n",
    "intervalle par locuteur. **Contrôle** : la cellule « sans / propre » doit être proche du F1 de\n",
    "validation de C11 (`chosen.validation_f1` de `training/results/c11_family_b_emotions.json`,\n",
    "≈ 0,745 ; la tête du registre vient de C13 et peut différer un peu) et d'un WER de même ordre\n",
    "que ceux de C17 sur `test` ; un grand écart signale une chaîne différente de celle de\n",
    "l'entraînement (à corriger avant de lire le reste)."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "from echolens.training.evaluation import macro_f1\n",
    "from echolens.training.evaluation import bootstrap_ci\n",
    "from echolens.training.transcription_eval import bootstrap_intervals\n",
    "\n",
    "emotion_targets = np.array([target for _, target in emotion_clips], dtype=np.int_)\n",
    "results = {}\n",
    "for strength in ablation.strengths:\n",
    "    for snr_db in ablation.snr_db:\n",
    "        key = cell_key(strength, snr_db)\n",
    "        cell = cells[key]\n",
    "        predictions = np.array(cell[\"emotion_predictions\"], dtype=np.int_)\n",
    "        f1 = macro_f1(emotion_targets, predictions)\n",
    "        f1_low, f1_high = bootstrap_ci(emotion_targets, predictions, evaluation_config, macro_f1)\n",
    "        scores = []\n",
    "        for row, hypothesis in zip(dev_rows, cell[\"hypotheses\"], strict=True):\n",
    "            scores.append(score_sentence(row.path, row.speaker, row.transcript, hypothesis))\n",
    "        wer, _ = bootstrap_intervals(\n",
    "            scores, transcription_config.bootstrap_draws, transcription_config.seed,\n",
    "            transcription_config.confidence_level,\n",
    "        )\n",
    "        results[key] = {\n",
    "            \"emotion_f1\": [f1, f1_low, f1_high],\n",
    "            \"wer\": [wer.value, wer.low, wer.high],\n",
    "        }\n",
    "\n",
    "import pandas as pd\n",
    "\n",
    "rows_for_table = []\n",
    "for key, value in results.items():\n",
    "    strength_name, snr_name = key.split(\"|\")\n",
    "    rows_for_table.append({\n",
    "        \"force\": strength_name,\n",
    "        \"condition\": snr_name if snr_name == \"clean\" else f\"{snr_name} dB\",\n",
    "        \"F1 émotions\": f\"{value['emotion_f1'][0]:.3f} [{value['emotion_f1'][1]:.3f} ; {value['emotion_f1'][2]:.3f}]\",\n",
    "        \"WER\": f\"{value['wer'][0]:.1%} [{value['wer'][1]:.1%} ; {value['wer'][2]:.1%}]\",\n",
    "    })\n",
    "table = pd.DataFrame(rows_for_table)\n",
    "display(table)\n",
    "table.to_csv(OUTPUT_DIR / \"c19_table.csv\", index=False)\n",
    "\n",
    "reference_key = cell_key(None, None)\n",
    "print(\"Contrôle « sans / propre » → F1 émotions\", round(results[reference_key][\"emotion_f1\"][0], 4),\n",
    "      \"· WER\", round(results[reference_key][\"wer\"][0], 4))\n",
    "c11 = json.loads((REPO_ROOT / \"training\" / \"results\" / \"c11_family_b_emotions.json\").read_text(encoding=\"utf-8\"))\n",
    "print(\"C11, F1 de validation de la tête retenue :\", c11[\"models\"][\"whisper-base\"][\"chosen\"])"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## h. Choix de la force (règle de D4)\n",
    "\n",
    "Score d'une force = moyenne, sur les quatre conditions, du F1 des émotions et de 1 − WER\n",
    "(plancher 0), puis moyenne des deux tâches. On affiche aussi la décision tâche par tâche : si\n",
    "elles divergent, c'est à noter dans le rapport (le débruitage peut aider la parole et abîmer les\n",
    "émotions, hypothèse de D4)."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "from echolens.training.denoise_ablation import choose_strength\n",
    "\n",
    "def strength_name(strength):\n",
    "    return \"none\" if strength is None else str(strength)\n",
    "\n",
    "emotion_mean = {}\n",
    "transcription_mean = {}\n",
    "for strength in ablation.strengths:\n",
    "    f1_values = []\n",
    "    wer_scores = []\n",
    "    for snr_db in ablation.snr_db:\n",
    "        value = results[cell_key(strength, snr_db)]\n",
    "        f1_values.append(value[\"emotion_f1\"][0])\n",
    "        wer_scores.append(max(0.0, 1.0 - value[\"wer\"][0]))\n",
    "    emotion_mean[strength_name(strength)] = float(np.mean(f1_values))\n",
    "    transcription_mean[strength_name(strength)] = float(np.mean(wer_scores))\n",
    "\n",
    "overall = {}\n",
    "for name in emotion_mean:\n",
    "    overall[name] = (emotion_mean[name] + transcription_mean[name]) / 2\n",
    "\n",
    "chosen, rounded = choose_strength(overall, ablation.tie_decimals)\n",
    "chosen_emotion, _ = choose_strength(emotion_mean, ablation.tie_decimals)\n",
    "chosen_transcription, _ = choose_strength(transcription_mean, ablation.tie_decimals)\n",
    "summary = pd.DataFrame({\n",
    "    \"F1 émotions (moy.)\": emotion_mean,\n",
    "    \"1 − WER (moy.)\": transcription_mean,\n",
    "    \"score moyen\": overall,\n",
    "}).round(4)\n",
    "display(summary)\n",
    "print(\"Force retenue (règle de D4) :\", chosen)\n",
    "print(\"  si on ne regardait que les émotions :\", chosen_emotion)\n",
    "print(\"  si on ne regardait que la transcription :\", chosen_transcription)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## i. Figure pour le rapport\n",
    "\n",
    "Score selon le niveau de bruit, une courbe par force : WER à gauche (plus bas = mieux), F1 des émotions à droite, barres = intervalles à 95 %."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "import matplotlib.pyplot as plt\n",
    "\n",
    "condition_names = [\"propre\" if snr is None else f\"{snr} dB\" for snr in ablation.snr_db]\n",
    "positions = np.arange(len(condition_names))\n",
    "figure, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)\n",
    "panels = [(\"wer\", \"WER (↓)\", axes[0]), (\"emotion_f1\", \"F1 macro des émotions (↑)\", axes[1])]\n",
    "for metric, title, axis in panels:\n",
    "    for index, strength in enumerate(ablation.strengths):\n",
    "        values = []\n",
    "        lows = []\n",
    "        highs = []\n",
    "        for snr_db in ablation.snr_db:\n",
    "            value, low, high = results[cell_key(strength, snr_db)][metric]\n",
    "            values.append(value)\n",
    "            lows.append(value - low)\n",
    "            highs.append(high - value)\n",
    "        shift = (index - 1.5) * 0.06\n",
    "        label = \"sans\" if strength is None else f\"force {strength}\"\n",
    "        axis.errorbar(positions + shift, values, yerr=[lows, highs], marker=\"o\", capsize=3, label=label)\n",
    "    axis.set_title(title)\n",
    "    axis.set_xticks(positions, condition_names)\n",
    "    axis.set_xlabel(\"niveau de bruit (SNR)\")\n",
    "    axis.grid(alpha=0.3)\n",
    "axes[0].legend()\n",
    "figure.tight_layout()\n",
    "FIGURE_PATH = OUTPUT_DIR / \"c19_denoising_ablation.png\"\n",
    "figure.savefig(FIGURE_PATH, dpi=200)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## j. Résultats et paragraphe pour le rapport\n",
    "\n",
    "`c19_denoising_ablation.json` (→ `training/results/`), `c19_denoising_ablation.png`\n",
    "(→ `report/figures/`). Le choix s'applique ensuite dans `configs/preprocess.yaml` :\n",
    "`enabled: true` et `prop_decrease: <force>` si une force gagne, `enabled: false` si « sans » gagne\n",
    "(avec le commentaire de la règle de D4 et ce résultat)."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "import datetime\n",
    "import platform\n",
    "import subprocess\n",
    "\n",
    "commit = subprocess.run(\n",
    "    [\"git\", \"-C\", CLONE_DIR, \"rev-parse\", \"HEAD\"], capture_output=True, text=True, check=True\n",
    ").stdout.strip()\n",
    "content = {\n",
    "    \"experiment\": \"c19_denoising_ablation\",\n",
    "    \"date\": datetime.datetime.now(datetime.UTC).isoformat(),\n",
    "    \"branch\": BRANCH,\n",
    "    \"commit\": commit,\n",
    "    \"python\": platform.python_version(),\n",
    "    \"gpu\": torch.cuda.get_device_name(0),\n",
    "    \"config\": ablation.model_dump(),\n",
    "    \"denoise_fixed\": {\n",
    "        \"stationary\": base_denoise_config.stationary,\n",
    "        \"n_std_thresh_stationary\": base_denoise_config.n_std_thresh_stationary,\n",
    "        \"n_fft\": base_denoise_config.n_fft,\n",
    "    },\n",
    "    \"emotion_model\": {\"id\": entry.id, \"revision\": entry.revision, \"validation_size\": len(emotion_clips)},\n",
    "    \"whisper\": {\"id\": whisper_model_config.id, \"revision\": whisper_model_config.revision, \"sentences\": len(dev_rows)},\n",
    "    \"cells\": results,\n",
    "    \"mean_by_strength\": {\"emotion_f1\": emotion_mean, \"one_minus_wer\": transcription_mean, \"overall\": overall},\n",
    "    \"chosen_strength\": chosen,\n",
    "    \"chosen_by_task\": {\"emotions\": chosen_emotion, \"transcription\": chosen_transcription},\n",
    "    \"runtime_s\": round(time.time() - NOTEBOOK_START),\n",
    "}\n",
    "results_path = OUTPUT_DIR / \"c19_denoising_ablation.json\"\n",
    "results_path.write_text(json.dumps(content, indent=2, ensure_ascii=False) + \"\\n\", encoding=\"utf-8\")\n",
    "print(results_path)\n",
    "\n",
    "best = results[cell_key(None, 0)]\n",
    "chosen_value = None if chosen == \"none\" else float(chosen)\n",
    "chosen_zero_db = results[cell_key(chosen_value, 0)]\n",
    "print()\n",
    "print(\"Brouillon de paragraphe (à relire et corriger) :\")\n",
    "print(\n",
    "    f\"L'ablation du débruitage (D4) compare quatre forces (sans, 0,5, 0,8, 1,0) sur la validation, \"\n",
    "    f\"propre et bruitée à 20, 10 et 0 dB. À 0 dB, sans débruitage : WER {best['wer'][0]:.1%}, \"\n",
    "    f\"F1 des émotions {best['emotion_f1'][0]:.3f} ; avec la force retenue ({chosen}) : \"\n",
    "    f\"WER {chosen_zero_db['wer'][0]:.1%}, F1 {chosen_zero_db['emotion_f1'][0]:.3f}. \"\n",
    "    f\"La règle de D4 (meilleur score moyen, la plus faible force en cas d'égalité) retient : {chosen}.\"\n",
    ")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}